# Create Vilcek Foundation Awards (PRIZE PATTERN, WP REST wrapped envelope)

Ingest of all Vilcek Foundation prize programs honoring immigrants' contributions to American arts and sciences. Source: `vilcek.org` — WordPress with a **non-standard wrapped REST envelope** at `/wp-json/wp/v2/prize_recipients` returning `{success: true, data: {records: [...]}}` instead of the usual direct-array response. First ingest on the project to encounter this WP customization; documented for future contributors.

**Awarding body:** Vilcek Foundation — `F4320307087` (US, DOI `10.13039/100002374`).

**Programs covered** (verified 2026-05-22 from `/wp-json/wp/v2/prize_type`):

| Program | Rows | Amount |
|---|---|---|
| Vilcek Prize | 44 | $100,000 |
| Vilcek Prize for Creative Promise | 98 | $50,000 |
| Vilcek Prize for Excellence | 5 | $100,000 |
| Marica Vilcek Prize | 5 | $100,000 |
| Vilcek Prize for Creative Promise Honoree | 32 | NULL (§6.7 waiver) |
| **Total** | **184** | mix |

**Multi-scheme amount handling**: most pre-2024 records have empty `acf.prize_amount` strings. The script parses ACF first, then falls back to a per-scheme map (verified from each prize's own `/prizes/{slug}/` overview page on vilcek.org). 83% amount coverage post-fallback; the 32 NULL rows are honorees, which by program design carry no monetary award.

**Disambiguation note**: recipients carry multiple `prize_type` taxonomy tags (e.g., a Creative Promise winner is also a generic "Vilcek Prize Honoree"). The script picks the first NON-honoree-tagged term as the canonical `funder_scheme`.

**Corpus** (verified 2026-05-22 full local fetch): **184 recipients** spanning **2006-2026** (21 distinct years). 99% affiliation + description coverage. Country defaults to US (Vilcek prizes go to US-resident immigrants by program rule).

**Source authority** — `vilcek.org` is the foundation's own site. Method #2 (REST API) on the runbook ladder.

**S3 location**: `s3a://openalex-ingest/awards/vilcek/vilcek_prizes.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.vilcek_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/vilcek/vilcek_prizes.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.vilcek_raw;

## Step 1.5: Inspect raw + multi-scheme amount-tier coverage

**Verified coverage** (184 recipients, 2026-05-22): 100% name/year/scheme/slug/landing; 99% affiliation/description; 83% amount (NULL on 32 Honoree rows by program design, §6.7-waived per-scheme).

In [ ]:
%sql
DESCRIBE openalex.awards.vilcek_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.vilcek_raw LIMIT 5;

In [ ]:
%sql
-- §1.5 multi-scheme amount scan — confirms tiers + 5th-scheme NULL.
SELECT scheme,
       COUNT(*) AS rows,
       COUNT(amount) AS has_amount,
       MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
       MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount
FROM openalex.awards.vilcek_raw
GROUP BY scheme
ORDER BY rows DESC;

## Step 1.6: Fail-fast — verify Vilcek Foundation funder row exists

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320307087;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.vilcek_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320307087  -- Vilcek Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    n.display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'prize' AS funding_type,
    n.scheme AS funder_scheme,
    'vilcek_foundation' AS provenance,
    TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(n.end_date,   'yyyy-MM-dd') AS end_date,
    TRY_CAST(SUBSTRING(n.start_date, 1, 4) AS INT) AS start_year,
    TRY_CAST(SUBSTRING(n.end_date,   1, 4) AS INT) AS end_year,
    CASE
        WHEN n.name IS NULL OR n.name = '' THEN NULL
        ELSE struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS role_start,
            struct(
                n.affiliation AS name,
                n.country AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    n.declined,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.vilcek_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.name IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 105

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'vilcek_foundation' AND priority = 105;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    105 AS priority  -- Vilcek priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.vilcek_awards;

## Step 6: Verification

- Row count: **184** (44 Vilcek Prize + 98 Creative Promise + 5 Excellence + 5 Marica + 32 Honoree)
- §6.7 partial-waiver: 32 Honoree rows ship amount=NULL (no monetary award by program design)
- 21 distinct years (2006-2026)

In [ ]:
%sql
SELECT COUNT(*) AS total_vilcek_award_rows FROM openalex.awards.vilcek_awards;

In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.vilcek_awards;

In [ ]:
%sql
SELECT funder_scheme,
       COUNT(*) AS total,
       COUNT(amount) AS has_amount,
       ROUND(MIN(amount), 0) AS min_amount,
       ROUND(MAX(amount), 0) AS max_amount
FROM openalex.awards.vilcek_awards
GROUP BY funder_scheme
ORDER BY total DESC;

In [ ]:
%sql
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.vilcek_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;